# Project 3: Name Gender Classifier
## DATA 620
## Miraj Patel

In [3]:
import nltk
import random
from nltk.corpus import names
from nltk.classify import apply_features
import pandas as pd
import matplotlib.pyplot as plt

# Download the names corpus
nltk.download('names')

[nltk_data] Downloading package names to
[nltk_data]     C:\Users\Miraj\AppData\Roaming\nltk_data...
[nltk_data]   Package names is already up-to-date!


True

In [4]:
# Load data
# Use a seed for reproducibility
labeled_names = ([(name, 'male') for name in names.words('male.txt')] +
                 [(name, 'female') for name in names.words('female.txt')])
random.seed(42)
random.shuffle(labeled_names)

print(f"Total names in corpus: {len(labeled_names)}")

Total names in corpus: 7944


**Project Background**

The objective of this project is to build a supervised machine learning classifier to predict the gender of a name based on its orthographic features. Following the guidelines from Natural Language Processing with Python, I am splitting the 7,944 names into three distinct subsets:

*Test Set (500):* Final evaluation to check real-world performance.

*Dev-Test Set (500):* Used for error analysis and feature refinement.

*Training Set (6,944):* The primary data used to teach the model.

In [5]:
# Splitting per guidelines
test_names = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names = labeled_names[1000:]

print(f"Training set: {len(train_names)}")
print(f"Dev-Test set: {len(devtest_names)}")
print(f"Test set: {len(test_names)}")

Training set: 6944
Dev-Test set: 500
Test set: 500


**Feature Engineering**

I will start with a simple baseline and incrementally improve the feautre set. By analyzing the Dev-Test errors, I can identify which linguistic patterns, such as suffixes, prefixes, or length, provide the most information gain for the classifier.

Step 1: build our Baseline Classifier. Let's start with the simplest feature for the gender classifier, the last letter of the name. In English names, this single character often carries significant gender information. An example would be names ending in "a" are statistically more likely to be female.

In [10]:
# Define the simplest feature extractor (Baseline)
def gender_features_v1(word):
    return {'last_letter': word[-1].lower()}

# Apply features to our pre-defined splits
train_set_v1 = [(gender_features_v1(n), g) for (n, g) in train_names]
devtest_set_v1 = [(gender_features_v1(n), g) for (n, g) in devtest_names]
test_set_v1 = [(gender_features_v1(n), g) for (n, g) in test_names]

# Train the Naive Bayes Classifier
classifier_v1 = nltk.NaiveBayesClassifier.train(train_set_v1)

# Evaluate baseline accuracy
accuracy_v1 = nltk.classify.accuracy(classifier_v1, devtest_set_v1)
print(f"Step 1. Baseline Accuracy (Dev-Test): {accuracy_v1:.4f}")

# Show the most informative features
classifier_v1.show_most_informative_features(10)

Step 1. Baseline Accuracy (Dev-Test): 0.7540
Most Informative Features
             last_letter = 'a'            female : male   =     31.8 : 1.0
             last_letter = 'k'              male : female =     28.6 : 1.0
             last_letter = 'f'              male : female =     26.8 : 1.0
             last_letter = 'p'              male : female =     12.0 : 1.0
             last_letter = 'd'              male : female =      9.7 : 1.0
             last_letter = 'o'              male : female =      8.9 : 1.0
             last_letter = 'm'              male : female =      8.7 : 1.0
             last_letter = 'v'              male : female =      8.6 : 1.0
             last_letter = 'r'              male : female =      7.0 : 1.0
             last_letter = 'g'              male : female =      5.1 : 1.0


The baseline classifier, utilizing only the final character of each name, achieved an accuracy of **75.40%** on the Dev-Test set. This demonstrates that alphabet endings are a significant gender indicator in English names, but this single feature is insufficient for high-precision classification.


Step 2: expand the feature set to include the last two letters, the first letter and the length of the name. 

In [11]:
# Define an improved feature extractor (v2)
def gender_features_v2(name):
    name = name.lower()
    features = {
        'first_letter': name[0],
        'last_letter': name[-1],
        'last_two_letters': name[-2:],
        'length': len(name)
    }
    return features

# Apply step 2 features to our splits
train_set_v2 = [(gender_features_v2(n), g) for (n, g) in train_names]
devtest_set_v2 = [(gender_features_v2(n), g) for (n, g) in devtest_names]

# Train the improved classifier
classifier_v2 = nltk.NaiveBayesClassifier.train(train_set_v2)

# Evaluate improved accuracy
accuracy_v2 = nltk.classify.accuracy(classifier_v2, devtest_set_v2)
print(f"Step 2. Improved Accuracy (v2) (Dev-Test): {accuracy_v2:.4f}")

# Show changes in the informative features
classifier_v2.show_most_informative_features(10)

Step 2. Improved Accuracy (v2) (Dev-Test): 0.7840
Most Informative Features
        last_two_letters = 'na'           female : male   =     93.9 : 1.0
        last_two_letters = 'la'           female : male   =     70.8 : 1.0
        last_two_letters = 'rd'             male : female =     38.7 : 1.0
        last_two_letters = 'ia'           female : male   =     35.9 : 1.0
        last_two_letters = 'sa'           female : male   =     32.7 : 1.0
             last_letter = 'a'            female : male   =     31.8 : 1.0
        last_two_letters = 'ta'           female : male   =     29.3 : 1.0
             last_letter = 'k'              male : female =     28.6 : 1.0
        last_two_letters = 'us'             male : female =     27.6 : 1.0
             last_letter = 'f'              male : female =     26.8 : 1.0


Expanding the feature set to include the last two letters, first letter, and name length resulted in an accuracy of **78.40%**. This incremental improvement confirms that local context around the word ending provides much stronger predictive power than a single character.

Step 3: expand the feature set to in vowels counts and specififc character clusters. Example, female names often contain more vowels or specific combinations like "ee" or "ia". 

In [12]:
# Define an advanced feature extractor (v3)
def gender_features_v3(name):
    name = name.lower()
    features = {
        'first_letter': name[0],
        'last_letter': name[-1],
        'last_two_letters': name[-2:],
        'last_three_letters': name[-3:], # Adding 3-letter suffixes
        'length': len(name),
        'vowel_count': sum(1 for char in name if char in 'aeiouy'),
        'vowel_ratio': sum(1 for char in name if char in 'aeiouy') / len(name)
    }
    return features

# Apply step 3 features
train_set_v3 = [(gender_features_v3(n), g) for (n, g) in train_names]
devtest_set_v3 = [(gender_features_v3(n), g) for (n, g) in devtest_names]
test_set_v3 = [(gender_features_v3(n), g) for (n, g) in test_names]

# Train and evaluate
classifier_v3 = nltk.NaiveBayesClassifier.train(train_set_v3)
accuracy_v3 = nltk.classify.accuracy(classifier_v3, devtest_set_v3)

print(f"Step 3. Advanced Accuracy (v3) (Dev-Test): {accuracy_v3:.4f}")
classifier_v3.show_most_informative_features(10)

Step 3. Advanced Accuracy (v3) (Dev-Test): 0.8000
Most Informative Features
        last_two_letters = 'na'           female : male   =     93.9 : 1.0
        last_two_letters = 'la'           female : male   =     70.8 : 1.0
      last_three_letters = 'ard'            male : female =     43.4 : 1.0
        last_two_letters = 'rd'             male : female =     38.7 : 1.0
        last_two_letters = 'ia'           female : male   =     35.9 : 1.0
        last_two_letters = 'sa'           female : male   =     32.7 : 1.0
             last_letter = 'a'            female : male   =     31.8 : 1.0
        last_two_letters = 'ta'           female : male   =     29.3 : 1.0
      last_three_letters = 'nne'          female : male   =     29.1 : 1.0
             last_letter = 'k'              male : female =     28.6 : 1.0


The final incremental improvement, which introduced three-letter suffixes and vowel metrics, successfully pushed the classifier accuracy to **80.00%** on the Dev-Test set. This represents a cumulative improvement of **4.6%** over the baseline.

**Classifier Performance on Test Data**

We will check our final performance of our classifier on the *Test Set* from earlier. We will compare this to the Dev-Test performance to check for underfitting or overfitting.

In [13]:
# Evaluate the final model on the Test Set
final_test_accuracy = nltk.classify.accuracy(classifier_v3, test_set_v3)

print(f"Final Accuracy on Test Set: {final_test_accuracy:.4f}")
print(f"Final Accuracy on Dev-Test Set: {accuracy_v3:.4f}")

# Comparison Logic
diff = final_test_accuracy - accuracy_v3
print(f"Difference in performance: {diff:.4f}")

Final Accuracy on Test Set: 0.7940
Final Accuracy on Dev-Test Set: 0.8000
Difference in performance: -0.0060


**Comparison of Test vs. Dev-Test Performance**

The final evaluation on the held-out Test Set yielded an accuracy of *79.40%*, compared to *80.00%* on the Dev-Test set. This marginal difference of *0.6%* provides critical insight into the classifier's reliability.